# 04 — MD-style stability summary (lightweight proxy)

**Read this before running:** real molecular dynamics validation means solvating
a protein-ligand complex and running nanoseconds of production MD in GROMACS,
AMBER, or OpenMM with a parameterized force field — that's a specialist task,
not something that fits a free, hosted, single-click notebook. What this
notebook does instead is a fast, honest **first-pass filter**: it generates
many 3D conformers of each ligand, minimizes them, and measures how much the
molecule's shape varies (RMSD) around its lowest-energy conformation. A
compound whose low-energy conformers cluster tightly is a more rigid,
predictable binder than one that sprawls across many very different shapes —
a real signal, just not a substitute for true complex MD.

**Use this to cut your list down**, then send only the surviving compounds
for real MD (or a wet-lab assay) rather than treating this as a final answer.

Upload a CSV with a `SMILES` column — e.g. exported from the Streamlit app's
Screening or Docking tab.


In [ ]:
!pip install -q rdkit pandas numpy


In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

from google.colab import files
uploaded = files.upload()
input_file = list(uploaded.keys())[0]
df = pd.read_csv(input_file)
assert "SMILES" in df.columns, "Input CSV must have a SMILES column"
print(f"Loaded {len(df)} compounds")


In [ ]:
N_CONFORMERS = 20
RIGID_RMSD_THRESHOLD_A = 1.5  # angstroms -- tune to your own standard

def conformational_rmsd(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.nan
    mol = Chem.AddHs(mol)
    cids = AllChem.EmbedMultipleConfs(mol, numConfs=N_CONFORMERS, randomSeed=42)
    if len(cids) == 0:
        return np.nan

    energies = []
    for cid in cids:
        props = AllChem.MMFFGetMoleculeProperties(mol)
        ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
        if ff is None:
            continue
        ff.Minimize()
        energies.append((cid, ff.CalcEnergy()))

    if not energies:
        return np.nan

    best_cid = min(energies, key=lambda x: x[1])[0]
    rmsds = [
        AllChem.GetConformerRMS(mol, best_cid, cid)
        for cid, _ in energies if cid != best_cid
    ]
    return float(np.mean(rmsds)) if rmsds else 0.0


In [ ]:
df["conformational_rmsd_A"] = df["SMILES"].apply(conformational_rmsd)
df["md_verdict"] = np.where(
    df["conformational_rmsd_A"] <= RIGID_RMSD_THRESHOLD_A,
    "rigid / consistent shape",
    "flexible / uncertain pose",
)
df[["SMILES", "conformational_rmsd_A", "md_verdict"]]


In [ ]:
out = df[["SMILES", "conformational_rmsd_A", "md_verdict"]]
out.to_csv("md_summary.csv", index=False)
print("Saved md_summary.csv")

files.download("md_summary.csv")


**Next step:** upload `md_summary.csv` in the Streamlit app's **MD summary** tab — it merges these two columns into the shared results table next to the docking and potency scores.